# Phase 0 : Exploratory Data Analysis (EDA) 🔍

Objectif : Comprendre le dataset, la distribution de la cible `is_fraud`, et identifier les features les plus discriminantes.

*Note : Ce notebook utilise le DataLoader du projet.*

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from loguru import logger

# Configuration visuelle
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)

# Suppression des warnings inutiles
import warnings
warnings.filterwarnings('ignore')

## 1. Chargement des données

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent)) # Ajouter le dossier racine au path

from src.data.loader import DataLoader

loader = DataLoader()
df = loader.load()

df.head()

## 2. Distribution de la variable cible (`is_fraud`)

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x='is_fraud')
plt.title("Distribution des Transactions (Légitimes vs Fraude)")

# Ajouter les pourcentages
total = len(df)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.2f}%'
    x = p.get_x() + p.get_width() / 2 - 0.05
    y = p.get_height() + 100
    ax.annotate(percentage, (x, y), ha='center')

plt.show()

## 3. Analyse des Features Numériques
Regardons le `transaction_amount` (montant).

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='is_fraud', y='transaction_amount')
plt.yscale("log") # Log scale car les montants de fraude peuvent être extrêmes
plt.title("Distribution des Montants par class (Log Scale)")
plt.show()

## 4. Analyse des Variables Catégorielles

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Type de transaction
sns.countplot(data=df, x='transaction_type', hue='is_fraud', ax=axes[0])
axes[0].set_title("Fraudes par Type de Transaction")
axes[0].tick_params(axis='x', rotation=45)

# KYC Status
sns.countplot(data=df, x='kyc_verified', hue='is_fraud', ax=axes[1])
axes[1].set_title("Fraudes par Statut KYC")

plt.tight_layout()
plt.show()

## 5. Matrice de Corrélation Numérique

In [ ]:
num_cols = df.select_dtypes(include=['number', 'bool']).columns
corr = df[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdBu_r', fmt=".2f", vmin=-1, vmax=1)
plt.title("Matrice de Corrélation")
plt.show()

## 6. Vérification des Splits (Train / Val / Test)

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = loader.get_splits(df)

print(f"Train : {len(X_train)} (Fraudes : {y_train.sum()})")
print(f"Val   : {len(X_val)} (Fraudes : {y_val.sum()})")
print(f"Test  : {len(X_test)} (Fraudes : {y_test.sum()})")